 ## Imports and Setup

In [17]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from functools import partial
import multiprocessing
from time import time as timer
from tqdm import tqdm
import requests
import urllib
import warnings
warnings.filterwarnings('ignore')

# ML imports
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold, StratifiedKFold, TimeSeriesSplit, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import catboost as cb
import joblib

# Deep Learning imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

# Scipy for statistics
from scipy import stats
from typing import List, Dict, Any, Tuple

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Utility Functions

In [18]:
def download_image(image_link, savefolder):
    """Download a single image from URL"""
    if isinstance(image_link, str):
        filename = Path(image_link).name
        image_save_path = os.path.join(savefolder, filename)
        if not os.path.exists(image_save_path):
            try:
                urllib.request.urlretrieve(image_link, image_save_path)    
            except Exception as ex:
                print('Warning: Not able to download - {}\n{}'.format(image_link, ex))
        else:
            return
    return

def download_images(image_links, download_folder):
    """Download multiple images in parallel"""
    if not os.path.exists(download_folder):
        os.makedirs(download_folder)
    results = []
    download_image_partial = partial(download_image, savefolder=download_folder)
    with multiprocessing.Pool(100) as pool:
        for result in tqdm(pool.imap(download_image_partial, image_links), total=len(image_links)):
            results.append(result)
        pool.close()
        pool.join()

def load_and_preprocess_image(image_path, transform=None):
    """Load and preprocess image for model input"""
    try:
        image = Image.open(image_path).convert('RGB')
        if transform:
            image = transform(image)
        return image
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        if transform:
            blank_image = Image.new('RGB', (224, 224), color='white')
            return transform(blank_image)
        return None

def clean_text_advanced(text):
    """Advanced text cleaning for product descriptions"""
    if pd.isna(text) or text == '':
        return ''
    
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s\-\.\,\(\)]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def extract_price_features(text):
    """Extract price-related features from text"""
    features = {}
    
    # Look for quantity indicators
    qty_patterns = [r'pack of (\d+)', r'(\d+) pack', r'(\d+)x', r'x(\d+)', r'(\d+) pieces']
    for pattern in qty_patterns:
        match = re.search(pattern, text.lower())
        if match:
            features['quantity'] = int(match.group(1))
            break
    else:
        features['quantity'] = 1
    
    # Look for size/weight indicators
    size_patterns = [r'(\d+(?:\.\d+)?)\s*(oz|fl oz|g|kg|ml|l|lb)', r'(\d+(?:\.\d+)?)\s*(inch|in|cm|mm)']
    for pattern in size_patterns:
        match = re.search(pattern, text.lower())
        if match:
            features['size'] = float(match.group(1))
            features['unit'] = match.group(2)
            break
    else:
        features['size'] = 0
        features['unit'] = 'unknown'
    
    return features

## Data Loading and Preprocessing

In [3]:
import os
if os.path.exists('dataset/processed_subset/'):
    DATASET_FOLDER = 'dataset/processed_subset/'
elif os.path.exists('../dataset/processed_subset/'):
    DATASET_FOLDER = '../dataset/processed_subset/'
else:
    raise FileNotFoundError("Cannot find dataset folder. Please ensure you're running from the correct directory.")

print(f"Using dataset folder: {DATASET_FOLDER}")
train = pd.read_csv(os.path.join(DATASET_FOLDER, 'train_clean_subset.csv'))
test = pd.read_csv(os.path.join(DATASET_FOLDER, 'test_clean_subset.csv'))

print("DATA LOADED SUCCESSFULLY")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Using dataset folder: ../dataset/processed_subset/


DATA LOADED SUCCESSFULLY
Train shape: (18750, 5)
Test shape: (18750, 4)


In [4]:
# Data cleaning
train.drop_duplicates(inplace=True)
test.drop_duplicates(inplace=True)
train.dropna(subset=['price'], inplace=True)
train['price'] = pd.to_numeric(train['price'], errors='coerce')

# Fill missing values
train['catalog_content'].fillna('Unknown', inplace=True)
test['catalog_content'].fillna('Unknown', inplace=True)

# Combine for feature engineering
combined_df = pd.concat([train, test], ignore_index=True)

## Feature Engineering

In [ ]:
# Extract Brand Name
brand_pattern = r'Item Name: (\w+)'
combined_df['brand'] = combined_df['catalog_content'].str.extract(brand_pattern, expand=False).fillna('unknown')

# Extract Item Pack Quantity (IPQ)
ipq_pattern = r'(\d+)\s*(?:pack|count|oz|fl oz|g|kg|ml|l)'
combined_df['IPQ'] = combined_df['catalog_content'].str.extract(ipq_pattern, flags=re.IGNORECASE).fillna(1)
combined_df['IPQ'] = pd.to_numeric(combined_df['IPQ'], errors='coerce').fillna(1).astype(int)

# Create text-based features
combined_df['word_count'] = combined_df['catalog_content'].apply(lambda x: len(str(x).split()))
combined_df['char_count'] = combined_df['catalog_content'].apply(lambda x: len(str(x)))
combined_df['clean_text'] = combined_df['catalog_content'].apply(clean_text_advanced)

# Advanced price features
price_features = combined_df['catalog_content'].apply(extract_price_features)
combined_df['quantity'] = [f.get('quantity', 1) for f in price_features]
combined_df['size'] = [f.get('size', 0) for f in price_features]

print("Feature engineering completed!")

Feature engineering completed!


In [ ]:
# Split back to train/test
train_processed = combined_df[combined_df['price'].notna()].copy()
test_processed = combined_df[combined_df['price'].isna()].copy()

print(f"Processed train shape: {train_processed.shape}")
print(f"Processed test shape: {test_processed.shape}")

Processed train shape: (18750, 11)
Processed test shape: (18750, 11)


## Model Definitions

In [ ]:
class TextModel(nn.Module):
    """Neural network for text-based features"""
    def __init__(self, input_dim=300, hidden_dims=[256, 128, 64], dropout=0.3):
        super(TextModel, self).__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_dim),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, 1))
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

class ImageModel(nn.Module):
    """CNN model for image features using pretrained ResNet"""
    def __init__(self, pretrained=True, freeze_backbone=True):
        super(ImageModel, self).__init__()
        self.backbone = models.resnet50(weights="IMAGENET1K_V1" if pretrained else None)
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        # Replace final layer
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )
    
    def forward(self, x):
        return self.backbone(x)

class MultimodalModel(nn.Module):
    """Combined text and image model"""
    def __init__(self, text_dim=300, image_model_path=None):
        super(MultimodalModel, self).__init__()
        self.text_model = TextModel(text_dim, hidden_dims=[256, 128])
        self.image_model = ImageModel()
        
        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    
    def forward(self, text_features, images):
        text_out = self.text_model(text_features)
        image_out = self.image_model(images)
        
        # Concatenate outputs
        combined = torch.cat([text_out, image_out], dim=1)
        return self.fusion(combined)

class TraditionalMLWrapper(BaseEstimator, RegressorMixin):
    """Wrapper for traditional ML models with consistent interface"""
    def __init__(self, model_type='xgboost', **kwargs):
        self.model_type = model_type
        self.kwargs = kwargs
        self.model = None
        self._init_model()
    
    def _init_model(self):
        if self.model_type == 'xgboost':
            self.model = xgb.XGBRegressor(
                n_estimators=self.kwargs.get('n_estimators', 200),
                learning_rate=self.kwargs.get('learning_rate', 0.1),
                max_depth=self.kwargs.get('max_depth', 6),
                random_state=42
            )
        elif self.model_type == 'catboost':
            self.model = cb.CatBoostRegressor(
                iterations=self.kwargs.get('iterations', 200),
                learning_rate=self.kwargs.get('learning_rate', 0.1),
                depth=self.kwargs.get('depth', 6),
                verbose=False,
                random_state=42
            )
        elif self.model_type == 'random_forest':
            self.model = RandomForestRegressor(
                n_estimators=self.kwargs.get('n_estimators', 200),
                max_depth=self.kwargs.get('max_depth', 10),
                random_state=42
            )
        elif self.model_type == 'ridge':
            self.model = Ridge(
                alpha=self.kwargs.get('alpha', 1.0),
                random_state=42
            )
        else:
            raise ValueError(f"Unsupported model type: {self.model_type}")
    
    def fit(self, X, y):
        self.model.fit(X, y)
        return self
    
    def predict(self, X):
        return self.model.predict(X)
    
    def get_feature_importance(self):
        if hasattr(self.model, 'feature_importances_'):
            return self.model.feature_importances_
        elif hasattr(self.model, 'coef_'):
            return np.abs(self.model.coef_)
        else:
            return None

## Validation and Metrics

In [ ]:
def smape(y_true, y_pred):
    """Symmetric Mean Absolute Percentage Error"""
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

def evaluate_model(y_true, y_pred, model_name="Model"):
    """Comprehensive model evaluation"""
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    smape_score = smape(y_true, y_pred)
    
    metrics = {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'SMAPE': smape_score
    }
    
    print(f"\n{model_name} Performance:")
    print("-" * 30)
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    
    return metrics

class CrossValidator:
    """Advanced cross-validation with multiple strategies"""
    def __init__(self, cv_type='kfold', n_splits=5, random_state=42):
        self.cv_type = cv_type
        self.n_splits = n_splits
        self.random_state = random_state
        self.cv = self._get_cv_strategy()
    
    def _get_cv_strategy(self):
        if self.cv_type == 'kfold':
            return KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        elif self.cv_type == 'stratified':
            return StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        elif self.cv_type == 'timeseries':
            return TimeSeriesSplit(n_splits=self.n_splits)
        else:
            raise ValueError(f"Unsupported CV type: {self.cv_type}")
    
    def validate_model(self, model, X, y, scoring='smape'):
        """Perform cross-validation"""
        scores = []
        fold_predictions = []
        
        for fold, (train_idx, val_idx) in enumerate(self.cv.split(X, y)):
            X_train_fold, X_val_fold = X[train_idx], X[val_idx]
            y_train_fold, y_val_fold = y[train_idx], y[val_idx]
            
            # Clone and train model
            model_fold = clone(model)
            model_fold.fit(X_train_fold, y_train_fold)
            
            # Predict
            y_pred_fold = model_fold.predict(X_val_fold)
            
            # Calculate score
            if scoring == 'smape':
                score = smape(y_val_fold, y_pred_fold)
            elif scoring == 'mae':
                score = mean_absolute_error(y_val_fold, y_pred_fold)
            elif scoring == 'rmse':
                score = np.sqrt(mean_squared_error(y_val_fold, y_pred_fold))
            else:
                raise ValueError(f"Unsupported scoring: {scoring}")
            
            scores.append(score)
            fold_predictions.append((val_idx, y_pred_fold))
            
            print(f"Fold {fold + 1}: {scoring.upper()} = {score:.4f}")
        
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        
        print(f"\nCross-validation Results:")
        print(f"Mean {scoring.upper()}: {mean_score:.4f} (+/- {std_score:.4f})")
        
        return {
            'scores': scores,
            'mean_score': mean_score,
            'std_score': std_score,
            'fold_predictions': fold_predictions
        }

## Ensemble Methods

In [ ]:
class WeightedEnsemble(BaseEstimator, RegressorMixin):
    """Weighted average ensemble of multiple models"""
    def __init__(self, models, weights=None):
        self.models = models
        self.weights = weights or [1.0] * len(models)
        self.weights = np.array(self.weights) / np.sum(self.weights)
    
    def fit(self, X, y):
        for model in self.models:
            model.fit(X, y)
        return self
    
    def predict(self, X):
        predictions = np.array([model.predict(X) for model in self.models])
        return np.average(predictions, axis=0, weights=self.weights)

class StackingEnsemble(BaseEstimator, RegressorMixin):
    """Stacking ensemble with meta-learner"""
    def __init__(self, base_models, meta_model, cv=5):
        self.base_models = base_models
        self.meta_model = meta_model
        self.cv = cv
        self.trained_base_models = []
    
    def fit(self, X, y):
        # Generate meta-features using cross-validation
        meta_features = np.zeros((len(X), len(self.base_models)))
        
        kf = KFold(n_splits=self.cv, shuffle=True, random_state=42)
        
        for i, (train_idx, val_idx) in enumerate(kf.split(X)):
            X_train_fold, X_val_fold = X[train_idx], X[val_idx]
            y_train_fold = y[train_idx]
            
            fold_models = []
            for j, base_model in enumerate(self.base_models):
                model_clone = clone(base_model)
                model_clone.fit(X_train_fold, y_train_fold)
                fold_models.append(model_clone)
                
                # Predict on validation fold
                val_pred = model_clone.predict(X_val_fold)
                meta_features[val_idx, j] = val_pred
        
        # Train final base models on full data
        self.trained_base_models = []
        for base_model in self.base_models:
            model_clone = clone(base_model)
            model_clone.fit(X, y)
            self.trained_base_models.append(model_clone)
        
        # Train meta-model
        self.meta_model.fit(meta_features, y)
        return self
    
    def predict(self, X):
        # Get base model predictions
        base_predictions = np.array([model.predict(X) for model in self.trained_base_models]).T
        
        # Meta-model prediction
        return self.meta_model.predict(base_predictions)

class BlendingEnsemble(BaseEstimator, RegressorMixin):
    """Blending ensemble with holdout validation"""
    def __init__(self, base_models, meta_model, blend_ratio=0.2):
        self.base_models = base_models
        self.meta_model = meta_model
        self.blend_ratio = blend_ratio
        self.trained_base_models = []
    
    def fit(self, X, y):
        # Split data for blending
        X_blend, X_holdout, y_blend, y_holdout = train_test_split(
            X, y, test_size=self.blend_ratio, random_state=42
        )
        
        # Train base models on blend set
        self.trained_base_models = []
        holdout_predictions = []
        
        for base_model in self.base_models:
            model_clone = clone(base_model)
            model_clone.fit(X_blend, y_blend)
            self.trained_base_models.append(model_clone)
            
            # Predict on holdout
            holdout_pred = model_clone.predict(X_holdout)
            holdout_predictions.append(holdout_pred)
        
        # Prepare meta-features
        meta_features = np.array(holdout_predictions).T
        
        # Train meta-model
        self.meta_model.fit(meta_features, y_holdout)
        return self
    
    def predict(self, X):
        # Get base model predictions
        base_predictions = np.array([model.predict(X) for model in self.trained_base_models]).T
        
        # Meta-model prediction
        return self.meta_model.predict(base_predictions)

class AdaptiveEnsemble(BaseEstimator, RegressorMixin):
    """Adaptive ensemble that adjusts weights based on performance"""
    def __init__(self, models, adaptation_rate=0.1):
        self.models = models
        self.adaptation_rate = adaptation_rate
        self.weights = np.ones(len(models)) / len(models)
        self.performance_history = []
    
    def fit(self, X, y):
        for model in self.models:
            model.fit(X, y)
        return self
    
    def predict(self, X):
        predictions = np.array([model.predict(X) for model in self.models])
        return np.average(predictions, axis=0, weights=self.weights)
    
    def update_weights(self, y_true, X_val):
        """Update weights based on validation performance"""
        predictions = [model.predict(X_val) for model in self.models]
        errors = [mean_absolute_error(y_true, pred) for pred in predictions]
        
        # Convert errors to weights (lower error = higher weight)
        inv_errors = 1.0 / (np.array(errors) + 1e-8)
        new_weights = inv_errors / np.sum(inv_errors)
        
        # Adaptive update
        self.weights = (1 - self.adaptation_rate) * self.weights + self.adaptation_rate * new_weights
        self.performance_history.append(errors)

## Complete Pipeline Class

In [ ]:
class ComprehensivePricingPipeline:
    """Complete end-to-end pricing prediction pipeline"""
    
    def __init__(self, config=None):
        self.config = config or self._get_default_config()
        self.models = {}
        self.ensemble = None
        self.feature_columns = None
        self.tfidf_vectorizer = None
        self.device = device
        self.image_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
    def _get_default_config(self):
        return {
            "models": {
                "xgboost": {"n_estimators": 200, "learning_rate": 0.1},
                "catboost": {"iterations": 200, "learning_rate": 0.1},
                "random_forest": {"n_estimators": 200, "max_depth": 10},
                "ridge": {"alpha": 1.0}
            },
            "ensemble_type": "weighted",
            "use_text_features": True,
            "use_image_features": False,
            "text_feature_dim": 300,
            "cv_folds": 5
        }
    
    def prepare_features(self, df, fit_tfidf=False):
        """Prepare features for training/prediction"""
        # Basic numerical features
        feature_cols = ['IPQ', 'word_count', 'char_count', 'quantity', 'size']
        X_basic = df[feature_cols].fillna(0).values
        
        # Text features using TF-IDF
        if self.config["use_text_features"]:
            if fit_tfidf:
                self.tfidf_vectorizer = TfidfVectorizer(
                    max_features=self.config["text_feature_dim"],
                    stop_words='english',
                    ngram_range=(1, 2)
                )
                X_text = self.tfidf_vectorizer.fit_transform(df['clean_text']).toarray()
            else:
                if self.tfidf_vectorizer is None:
                    raise ValueError("TF-IDF vectorizer not fitted. Call with fit_tfidf=True first.")
                X_text = self.tfidf_vectorizer.transform(df['clean_text']).toarray()
            
            X_combined = np.hstack([X_basic, X_text])
        else:
            X_combined = X_basic
        
        return X_combined
    
    def train_models(self, X_train, y_train, X_val=None, y_val=None):
        """Train all configured models"""
        print("Training individual models...")
        
        for model_name, model_params in self.config["models"].items():
            print(f"Training {model_name}...")
            
            model = TraditionalMLWrapper(model_type=model_name, **model_params)
            model.fit(X_train, y_train)
            self.models[model_name] = model
            
            if X_val is not None and y_val is not None:
                val_pred = model.predict(X_val)
                metrics = evaluate_model(y_val, val_pred, f"{model_name}")
        
        print(f"Trained {len(self.models)} models successfully!")
    
    def create_ensemble(self, X_train, y_train):
        """Create ensemble from trained models"""
        base_models = list(self.models.values())
        
        if self.config["ensemble_type"] == "weighted":
            # Simple equal weights
            self.ensemble = WeightedEnsemble(base_models)
        elif self.config["ensemble_type"] == "stacking":
            meta_model = TraditionalMLWrapper('ridge')
            self.ensemble = StackingEnsemble(base_models, meta_model, cv=self.config["cv_folds"])
        elif self.config["ensemble_type"] == "blending":
            meta_model = TraditionalMLWrapper('ridge')
            self.ensemble = BlendingEnsemble(base_models, meta_model)
        elif self.config["ensemble_type"] == "adaptive":
            self.ensemble = AdaptiveEnsemble(base_models)
        else:
            raise ValueError(f"Unsupported ensemble type: {self.config['ensemble_type']}")
        
        print(f"Training {self.config['ensemble_type']} ensemble...")
        self.ensemble.fit(X_train, y_train)
        print("Ensemble training completed!")
    
    def predict(self, X):
        """Make predictions using the ensemble"""
        if self.ensemble is None:
            raise ValueError("Ensemble not trained. Call create_ensemble() first.")
        return self.ensemble.predict(X)
    
    def cross_validate(self, X, y, cv_type='kfold'):
        """Perform cross-validation"""
        validator = CrossValidator(cv_type=cv_type, n_splits=self.config["cv_folds"])
        
        # Validate individual models
        cv_results = {}
        for model_name, model in self.models.items():
            print(f"\nCross-validating {model_name}...")
            results = validator.validate_model(model, X, y, scoring='smape')
            cv_results[model_name] = results
        
        return cv_results

## Training and Execution Functions

In [ ]:
def quick_train_and_predict(config=None):
    
    # Initialize pipeline
    pipeline = ComprehensivePricingPipeline(config)
    
    # Prepare features
    print("Preparing features...")
    X_train = pipeline.prepare_features(train_processed, fit_tfidf=True)
    X_test = pipeline.prepare_features(test_processed, fit_tfidf=False)
    y_train = train_processed['price'].values
    
    # Split for validation
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )
    
    # Train models
    pipeline.train_models(X_train_split, y_train_split, X_val_split, y_val_split)
    
    # Create ensemble
    pipeline.create_ensemble(X_train_split, y_train_split)
    
    # Validate ensemble
    ensemble_pred = pipeline.predict(X_val_split)
    ensemble_metrics = evaluate_model(y_val_split, ensemble_pred, "Ensemble")
    
    # Generate predictions
    test_predictions = pipeline.predict(X_test)
    
    # Create submission
    submission_df = pd.DataFrame({
        "sample_id": test_processed["sample_id"],
        "price": test_predictions
    })
    
    submission_path = "submission_comprehensive.csv"
    submission_df.to_csv(submission_path, index=False)
    print(f"\n✅ Submission file created: {submission_path}")
    print(f"Predictions shape: {submission_df.shape}")
    
    return {
        'pipeline': pipeline,
        'ensemble_metrics': ensemble_metrics,
        'submission_df': submission_df
    }

def train_single_model(model_type='xgboost', config=None):
    """Train a single model quickly"""
    print(f"Training single {model_type} model...")
    
    # Prepare features
    feature_cols = ['IPQ', 'word_count', 'char_count', 'quantity', 'size']
    X_train = train_processed[feature_cols].fillna(0).values
    X_test = test_processed[feature_cols].fillna(0).values
    y_train = train_processed['price'].values
    
    # Split for validation
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )
    
    # Train model
    model = TraditionalMLWrapper(model_type=model_type)
    model.fit(X_train_split, y_train_split)
    
    # Validate
    val_pred = model.predict(X_val_split)
    metrics = evaluate_model(y_val_split, val_pred, f"{model_type}")
    
    # Generate predictions
    test_predictions = model.predict(X_test)
    
    # Create submission
    submission_df = pd.DataFrame({
        "sample_id": test_processed["sample_id"],
        "price": test_predictions
    })
    
    submission_path = f"submission_{model_type}.csv"
    submission_df.to_csv(submission_path, index=False)
    print(f"\n✅ Submission file created: {submission_path}")
    
    return {
        'model': model,
        'metrics': metrics,
        'submission_df': submission_df
    }

def advanced_training_with_cv(ensemble_type='stacking', cv_folds=5):
    """Advanced training with cross-validation"""
    print("Advanced Training Mode with Cross-Validation")
    print("=" * 50)
    
    config = {
        "models": {
            "xgboost": {"n_estimators": 300, "learning_rate": 0.08},
            "catboost": {"iterations": 300, "learning_rate": 0.08},
            "random_forest": {"n_estimators": 300, "max_depth": 12},
            "ridge": {"alpha": 0.5}
        },
        "ensemble_type": ensemble_type,
        "use_text_features": True,
        "text_feature_dim": 500,
        "cv_folds": cv_folds
    }
    
    pipeline = ComprehensivePricingPipeline(config)
    
    # Prepare features
    X_train = pipeline.prepare_features(train_processed, fit_tfidf=True)
    X_test = pipeline.prepare_features(test_processed, fit_tfidf=False)
    y_train = train_processed['price'].values
    
    # Train models
    pipeline.train_models(X_train, y_train)
    
    # Cross-validation
    cv_results = pipeline.cross_validate(X_train, y_train)
    
    # Create ensemble
    pipeline.create_ensemble(X_train, y_train)
    
    # Generate predictions
    test_predictions = pipeline.predict(X_test)
    
    # Create submission
    submission_df = pd.DataFrame({
        "sample_id": test_processed["sample_id"],
        "price": test_predictions
    })
    
    submission_path = f"submission_advanced_{ensemble_type}.csv"
    submission_df.to_csv(submission_path, index=False)
    print(f"\n✅ Advanced submission file created: {submission_path}")
    
    return {
        'pipeline': pipeline,
        'cv_results': cv_results,
        'submission_df': submission_df
    }

## Execution and Results

In [ ]:
# Basic execution - uncomment to run
# results = quick_train_and_predict()

Preparing features...


Training individual models...
Training xgboost...

xgboost Performance:
------------------------------
MAE: 14.4344
MSE: 498.2681
RMSE: 22.3219
R2: 0.2532
SMAPE: 67.3176
Training catboost...

catboost Performance:
------------------------------
MAE: 14.8060
MSE: 515.2603
RMSE: 22.6993
R2: 0.2277
SMAPE: 68.9199
Training random_forest...

random_forest Performance:
------------------------------
MAE: 14.8076
MSE: 527.2641
RMSE: 22.9622
R2: 0.2097
SMAPE: 69.8614
Training ridge...

ridge Performance:
------------------------------
MAE: 15.7446
MSE: 565.0099
RMSE: 23.7699
R2: 0.1532
SMAPE: 72.0208
Trained 4 models successfully!
Training weighted ensemble...
Ensemble training completed!

Ensemble Performance:
------------------------------
MAE: 14.6422
MSE: 506.2595
RMSE: 22.5002
R2: 0.2412
SMAPE: 68.6907

✅ Submission file created: submission_comprehensive.csv
Predictions shape: (18750, 2)


## Alternative Execution Modes

 Choose one of the following execution modes:

In [ ]:
# Mode 1: Quick training with default ensemble
# print("Mode 1: Quick Training")
# results_quick = quick_train_and_predict()

Mode 1: Quick Training
Preparing features...
Training individual models...
Training xgboost...

xgboost Performance:
------------------------------
MAE: 14.4344
MSE: 498.2681
RMSE: 22.3219
R2: 0.2532
SMAPE: 67.3176
Training catboost...

catboost Performance:
------------------------------
MAE: 14.8060
MSE: 515.2603
RMSE: 22.6993
R2: 0.2277
SMAPE: 68.9199
Training random_forest...

random_forest Performance:
------------------------------
MAE: 14.8076
MSE: 527.2641
RMSE: 22.9622
R2: 0.2097
SMAPE: 69.8614
Training ridge...

ridge Performance:
------------------------------
MAE: 15.7446
MSE: 565.0099
RMSE: 23.7699
R2: 0.1532
SMAPE: 72.0208
Trained 4 models successfully!
Training weighted ensemble...
Ensemble training completed!

Ensemble Performance:
------------------------------
MAE: 14.6422
MSE: 506.2595
RMSE: 22.5002
R2: 0.2412
SMAPE: 68.6907

✅ Submission file created: submission_comprehensive.csv
Predictions shape: (18750, 2)


In [ ]:
# Mode 2: Single model training
# print("\nMode 2: Single Model Training")
# results_single = train_single_model('xgboost')


Mode 2: Single Model Training
Training single xgboost model...

xgboost Performance:
------------------------------
MAE: 15.8761
MSE: 583.5484
RMSE: 24.1567
R2: 0.1254
SMAPE: 72.2860

✅ Submission file created: submission_xgboost.csv


In [15]:
# Mode 3: Advanced training with stacking ensemble
print("\nMode 3: Advanced Training with Stacking")
results_advanced = advanced_training_with_cv('stacking', cv_folds=3)


Mode 3: Advanced Training with Stacking
Advanced Training Mode with Cross-Validation
Training individual models...
Training xgboost...
Training catboost...
Training random_forest...
Training ridge...
Trained 4 models successfully!

Cross-validating xgboost...
Fold 1: SMAPE = 64.3225
Fold 2: SMAPE = 65.1911
Fold 3: SMAPE = 65.1207

Cross-validation Results:
Mean SMAPE: 64.8781 (+/- 0.3939)

Cross-validating catboost...
Fold 1: SMAPE = 66.4642
Fold 2: SMAPE = 67.1325
Fold 3: SMAPE = 66.6366

Cross-validation Results:
Mean SMAPE: 66.7444 (+/- 0.2832)

Cross-validating random_forest...
Fold 1: SMAPE = 68.6496
Fold 2: SMAPE = 69.3103
Fold 3: SMAPE = 68.9154

Cross-validation Results:
Mean SMAPE: 68.9584 (+/- 0.2714)

Cross-validating ridge...
Fold 1: SMAPE = 69.9547
Fold 2: SMAPE = 70.8698
Fold 3: SMAPE = 70.5547

Cross-validation Results:
Mean SMAPE: 70.4597 (+/- 0.3796)
Training stacking ensemble...
Ensemble training completed!

✅ Advanced submission file created: submission_advanced_sta

## Results Summary and Analysis

In [21]:
def analyze_results():
    """Analyze and compare all results"""
    print("\n" + "="*60)
    print("COMPREHENSIVE RESULTS ANALYSIS")
    print("="*60)
    
    print("\n1. Quick Training Results:")
    if 'results_quick' in locals():
        print(f"   - Ensemble SMAPE: {results_quick['ensemble_metrics']['SMAPE']:.4f}")
        print(f"   - Predictions generated: {len(results_quick['submission_df'])}")
    
    print("\n2. Single Model Results:")
    if 'results_single' in locals():
        print(f"   - XGBoost SMAPE: {results_single['metrics']['SMAPE']:.4f}")
        print(f"   - Predictions generated: {len(results_single['submission_df'])}")
    
    print("\n3. Advanced Training Results:")
    if 'results_advanced' in locals():
        print("   - Cross-validation completed")
        print(f"   - Predictions generated: {len(results_advanced['submission_df'])}")
    
    print("\n4. Files Generated:")
    import glob
    submission_files = glob.glob("submission_*.csv")
    for file in submission_files:
        print(f"   - {file}")
    
    print("All training modes completed successfully!")
    print("Choose the best performing submission file for final submission.")

analyze_results()


COMPREHENSIVE RESULTS ANALYSIS

1. Quick Training Results:

2. Single Model Results:

3. Advanced Training Results:

4. Files Generated:
   - submission_comprehensive.csv
   - submission_advanced_stacking.csv
   - submission_xgboost.csv
All training modes completed successfully!
Choose the best performing submission file for final submission.
